# KNN 후속 실험 시작 템플릿
현재 선정 모델을 검증한 후 사전등록한 후보만 실행한다. 예시 조건은 **계획이며 실행 결과가 아니다**. 개발 1,280장과 공식 Validation 19,200장만 평가. Original Test/교수님 Test 금지. 후보 실행은 기본 False. 환경 설정 및 입력은 공통 README 참조. 결과는 로컬 폴더에 저장하며 선택 모델을 바꾸지 않는다.

In [ ]:
from pathlib import Path
import sys, time, json, traceback
from datetime import datetime
import numpy as np
START=Path.cwd().resolve()
FT=next((p for p in (START,*START.parents) if (p/"00_Common"/"prepare.py").is_file()),None)
if FT is None: raise FileNotFoundError("Open this notebook within cloned 07_fine_tuning")
sys.path.insert(0,str(FT/"00_Common"))
from prepare import prepare, evaluate_followup
INPUT_DIR=Path.home()/"ml_fine_tuning_inputs" # edit
OUTPUT_DIR=Path.home()/"ml_fine_tuning_runs"/("KNN_next_"+datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
CURRENT_SVC=None # SVC only: set path of verified C2_G2 model.joblib
state=prepare("KNN",INPUT_DIR,OUTPUT_DIR,mode="followup",current_svc=CURRENT_SVC)
print("Baseline reproduced; local output:",OUTPUT_DIR)

## 후보 학습 및 집계 평가
아래 조건은 예시. 실행 전 계획을 확정하고 ID를 고유하게 부여한다. matched Batch1 시간은 별도로 측정하여 기록한다.

In [ ]:
RUN_CANDIDATE=False # enable only after preregistering the condition
EXPERIMENT_ID="FT-NEXT-KNN-001"
CONFIG={"pca_dims":96,"whiten":False,"k":3,"metric":"euclidean","weights":"distance","seed":42,"training_scope":"approved frozen training features only"}
if RUN_CANDIDATE:
    from sklearn.decomposition import PCA
    from sklearn.neighbors import KNeighborsClassifier
    log=OUTPUT_DIR/(EXPERIMENT_ID+"_fit.log")
    try:
        (OUTPUT_DIR/(EXPERIMENT_ID+"_plan.json")).write_text(json.dumps(CONFIG,indent=2))
        t=time.perf_counter()
        X=state["model"]._fit_X; y=state["model"]._y
        assert X.shape==(113920,1108)
        transform=PCA(n_components=CONFIG["pca_dims"],svd_solver="covariance_eigh",whiten=CONFIG["whiten"])
        z=transform.fit_transform(X)
        candidate=KNeighborsClassifier(n_neighbors=CONFIG["k"],metric=CONFIG["metric"],weights=CONFIG["weights"],algorithm="brute",n_jobs=2).fit(z,y)
        fit_s=time.perf_counter()-t
        log.write_text(f"FIT COMPLETED {fit_s}s\n")
        def predict_features(x): return candidate.predict(transform.transform(x))
        print(evaluate_followup(state,"KNN",EXPERIMENT_ID,predict_features,CONFIG,fit_s))
    except Exception:
        log.write_text(traceback.format_exc())
        raise
else:
    print("No candidate executed. Set RUN_CANDIDATE only after approval.")

## 기록
실측 metrics.csv/metrics.json, CM, 실행 로그, 환경과 계획을 보존한다. 별도 정식 시간 평가 및 팀 승인 없이는 최종 모델 교체 금지.